# Troupe 3D — dessin de planche → personnage 3D riggé (.glb)

Une exécution par personnage, sur un GPU gratuit (Kaggle T4/P100, Colab T4, Lightning L4). Voir `README.md`.

1. Déposez les dessins (pose **neutre** de chaque planche, fond vert ou détouré) dans `in/`.
2. Exécutez toutes les cellules. Résultat : `out/<nom>.glb` (riggé, humanoïde) + `out/<nom>.preview.png`.
3. Importez chaque `.glb` dans l'application (casting → « Importer un .vrm / .glb riggé »).

In [ ]:
import os, subprocess, json, glob, torch
VRAM = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
print(f'GPU: {torch.cuda.get_device_name(0) if VRAM else "aucun"} — {VRAM:.0f} Go')
# Le générateur de forme selon la mémoire : TRELLIS.2 (MIT) ≥ 24 Go, sinon Hunyuan3D-2.1 (forme 10 Go, texture 21 Go en low_vram).
SHAPE = 'trellis2' if VRAM >= 23 else 'hunyuan21'
RIG = 'mia'  # Make-It-Animatable (MIT, humanoïdes) ; 'unirig' en second choix
os.makedirs('in', exist_ok=True); os.makedirs('out', exist_ok=True); os.makedirs('work', exist_ok=True)
print('forme :', SHAPE, '| rig :', RIG)

In [ ]:
# 1) Dépendances (une fois par session)
!pip -q install rembg[gpu] trimesh pygltflib pillow numpy onnxruntime-gpu 2>/dev/null | tail -1
if SHAPE == 'trellis2':
    !git clone -q https://github.com/microsoft/TRELLIS.2 work/trellis2 2>/dev/null || true
    !cd work/trellis2 && pip -q install -r requirements.txt 2>/dev/null | tail -1
else:
    !git clone -q https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1 work/hunyuan21 2>/dev/null || true
    !cd work/hunyuan21 && pip -q install -r requirements.txt 2>/dev/null | tail -1
if RIG == 'mia':
    !git clone -q -b v2 https://github.com/jasongzy/Make-It-Animatable work/mia 2>/dev/null || true
else:
    !git clone -q https://github.com/VAST-AI-Research/UniRig work/unirig 2>/dev/null || true
print('ok')

In [ ]:
# 2) Découpe propre du dessin (le fond vert ou tout fond) → PNG avec alpha, cadré, 1024 px de haut
from rembg import remove, new_session
from PIL import Image
import numpy as np
sess = new_session('birefnet-general-lite')
cuts = {}
for f in sorted(glob.glob('in/*.*')):
    name = os.path.splitext(os.path.basename(f))[0]
    im = Image.open(f).convert('RGBA')
    out = remove(im, session=sess)
    a = np.asarray(out)[:, :, 3]
    ys, xs = np.where(a > 8)
    if len(ys) == 0:
        print('vide :', name); continue
    pad = int(0.04 * (ys.max() - ys.min()))
    box = (max(0, xs.min() - pad), max(0, ys.min() - pad), min(out.width, xs.max() + pad), min(out.height, ys.max() + pad))
    cut = out.crop(box)
    k = 1024 / cut.height
    cut = cut.resize((max(1, round(cut.width * k)), 1024), Image.LANCZOS)
    # Centré sur un carré (les générateurs attendent un sujet centré)
    sq = Image.new('RGBA', (1024, 1024), (0, 0, 0, 0)); sq.paste(cut, ((1024 - cut.width) // 2, 0))
    sq.save(f'work/{name}.cut.png'); cuts[name] = f'work/{name}.cut.png'
    print('découpé :', name, cut.size)
print(len(cuts), 'personnage(s)')

In [ ]:
# 3) Forme 3D texturée depuis le dessin
meshes = {}
if SHAPE == 'trellis2':
    import sys; sys.path.insert(0, 'work/trellis2')
    from trellis2.pipelines import Trellis2ImageTo3DPipeline  # voir le README du dépôt pour le nom exact du pipeline
    pipe = Trellis2ImageTo3DPipeline.from_pretrained('microsoft/TRELLIS.2-4B').cuda()
    for name, cut in cuts.items():
        mesh = pipe.run(Image.open(cut), seed=1)['mesh'][0]
        mesh.export(f'work/{name}.mesh.glb'); meshes[name] = f'work/{name}.mesh.glb'; print('forme :', name)
else:
    import sys; sys.path.insert(0, 'work/hunyuan21')
    from hy3dshape.pipelines import Hunyuan3DDiTFlowMatchingPipeline
    from hy3dpaint.textureGenPipeline import Hunyuan3DPaintPipeline, Hunyuan3DPaintConfig
    shape = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained('tencent/Hunyuan3D-2.1')
    for name, cut in cuts.items():
        mesh = shape(image=Image.open(cut))[0]
        mesh.export(f'work/{name}.shape.glb')
        print('forme :', name)
    del shape; torch.cuda.empty_cache()
    paint = Hunyuan3DPaintPipeline(Hunyuan3DPaintConfig(max_num_view=6, resolution=512))  # low VRAM: 6 vues, 512 px
    for name, cut in cuts.items():
        out = paint(mesh_path=f'work/{name}.shape.glb', image_path=cut, output_mesh_path=f'work/{name}.mesh.glb')
        meshes[name] = f'work/{name}.mesh.glb'; print('texture :', name)
print(len(meshes), 'maillage(s)')

In [ ]:
# 4) Rig humanoïde + poids de peau
rigged = {}
if RIG == 'mia':
    # Make-It-Animatable v2 : voir work/mia/README.md — l'application (app_v2.py) expose aussi une fonction d'inférence.
    for name, m in meshes.items():
        r = subprocess.run(['python', 'work/mia/app_v2.py', '--input', m, '--output', f'work/{name}.rigged.glb', '--no-gui'], capture_output=True, text=True)
        print(name, 'rig MIA', 'ok' if r.returncode == 0 else r.stderr[-400:])
        if r.returncode == 0: rigged[name] = f'work/{name}.rigged.glb'
else:
    for name, m in meshes.items():
        !cd work/unirig && bash launch/inference/generate_skeleton.sh --input ../../{m} --output ../../work/{name}.skel.fbx
        !cd work/unirig && bash launch/inference/generate_skin.sh --input ../../work/{name}.skel.fbx --output ../../work/{name}.skin.fbx
        !cd work/unirig && bash launch/inference/merge.sh --source ../../work/{name}.skin.fbx --target ../../{m} --output ../../work/{name}.rigged.glb
        if os.path.exists(f'work/{name}.rigged.glb'): rigged[name] = f'work/{name}.rigged.glb'
print(len(rigged), 'personnage(s) riggé(s)')

In [ ]:
# 5) Contrôle + export : squelette présent, hauteur normalisée, aperçu
from pygltflib import GLTF2
import shutil
for name, g in rigged.items():
    gl = GLTF2().load(g)
    joints = sum(len(s.joints) for s in (gl.skins or []))
    names = [n.name for n in gl.nodes if n.name]
    ok = joints >= 15
    print(f"{name}: {joints} os, exemples {names[:6]} → {'OK' if ok else 'rig insuffisant'}")
    if ok:
        shutil.copy(g, f'out/{name}.glb')
print('fichiers prêts :', sorted(os.listdir('out')))
# Le moteur reconnaît les os par leurs noms et la hiérarchie (rig3d.ts) : aucun renommage nécessaire.